# Verifiable AI Reasoning with Grok + xProof

Author: Jason Kensei

---

**What this cookbook demonstrates:** How to make Grok's reasoning cryptographically verifiable by anchoring decisions on-chain *before* output generation. Every Grok response gets a tamper-proof timestamp and hash — verifiable by anyone, irrefutable by design.

**Why it matters:** LLMs can say anything. xProof turns Grok outputs into verifiable artifacts — the reasoning hash is anchored on MultiversX blockchain before the output is delivered. This creates a 4W audit trail:
- **WHO** — Grok (model identity, version)
- **WHAT** — The output content hash
- **WHEN** — Blockchain timestamp (immutable)
- **WHY** — The reasoning/prompt hash anchored before generation

## Setup

You'll need:
1. An xAI API key from [console.x.ai](https://console.x.ai)
2. An xProof API key (free — 10 certifications included, no wallet needed)

### Get your free xProof API key

```bash
curl -X POST https://xproof.app/api/agent/register \
  -H "Content-Type: application/json" \
  -d '{"agent_name": "grok-cookbook-demo"}'
```

This returns an API key with 10 free certifications. No wallet or payment required.

In [ ]:
%pip install openai requests --quiet

In [ ]:
import hashlib
import json
import time
import requests
from openai import OpenAI

XAI_API_KEY = ""  # paste your xAI API key here
XPROOF_API_KEY = ""  # paste your xProof API key here
XPROOF_BASE = "https://xproof.app"

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

## Helper Functions

Two utilities:
1. `sha256_hash` — deterministic content hashing
2. `certify_on_xproof` — anchors a hash on MultiversX via xProof API

In [ ]:
def sha256_hash(content: str) -> str:
    """Generate SHA-256 hash of content."""
    return hashlib.sha256(content.encode("utf-8")).hexdigest()


def certify_on_xproof(file_hash: str, filename: str, metadata: dict) -> dict:
    """Certify a hash on xProof (MultiversX blockchain anchoring)."""
    response = requests.post(
        f"{XPROOF_BASE}/api/proof",
        headers={
            "Content-Type": "application/json",
            "x-api-key": XPROOF_API_KEY,
        },
        json={
            "fileHash": file_hash,
            "fileName": filename,
            "metadata": metadata,
        },
    )
    response.raise_for_status()
    return response.json()

## Step 1: Anchor the WHY (Intent) Before Generation

This is the key innovation. Before asking Grok to generate anything, we hash the prompt/intent and anchor it on-chain. This proves the reasoning existed *before* the output — not fabricated after the fact.

The `action_type: "reason"` tag in metadata marks this as a pre-execution anchor (WHY layer).

In [ ]:
prompt = "Analyze the current state of AI agent accountability. What are the key gaps in verifying that an AI agent did what it claims to have done?"

prompt_hash = sha256_hash(prompt)
print(f"Prompt hash: {prompt_hash[:16]}...")

why_proof = certify_on_xproof(
    file_hash=prompt_hash,
    filename="grok-reasoning-intent.txt",
    metadata={
        "xai_agent_id": "grok-cookbook-demo",
        "xai_model": "grok-3",
        "action_type": "reason",
        "stage": "pre_execution",
        "prompt_hash": prompt_hash,
    },
)

print(f"WHY proof anchored: {why_proof['id']}")
print(f"Verify: {XPROOF_BASE}/proof/{why_proof['id']}")

## Step 2: Generate with Grok

Now we call Grok. The prompt hash is already anchored on MultiversX — whatever Grok outputs, we can prove the intent preceded it.

In [ ]:
completion = client.chat.completions.create(
    model="grok-3",
    messages=[
        {
            "role": "system",
            "content": "You are a precise analyst. Give structured, evidence-based answers.",
        },
        {"role": "user", "content": prompt},
    ],
)

grok_output = completion.choices[0].message.content
grok_model = completion.model
print(f"Model: {grok_model}")
print(f"Output length: {len(grok_output)} chars")
print(f"\n{grok_output[:500]}...")

## Step 3: Anchor the WHAT (Output) After Generation

Now we hash and anchor the actual output. The metadata links back to the WHY proof via `prompt_hash`, creating a verifiable chain:

```
WHY (prompt hash, anchored BEFORE) → Grok generation → WHAT (output hash, anchored AFTER)
```

Anyone can verify that the intent existed before the output by checking the blockchain timestamps of both proofs.

In [ ]:
output_hash = sha256_hash(grok_output)
print(f"Output hash: {output_hash[:16]}...")

what_proof = certify_on_xproof(
    file_hash=output_hash,
    filename="grok-analysis-output.txt",
    metadata={
        "xai_agent_id": "grok-cookbook-demo",
        "xai_model": grok_model,
        "action_type": "generate",
        "stage": "post_execution",
        "prompt_hash": prompt_hash,
        "content_hash": output_hash,
    },
)

print(f"WHAT proof anchored: {what_proof['id']}")
print(f"Verify: {XPROOF_BASE}/proof/{what_proof['id']}")

## Step 4: Verify the Full 4W Audit Trail

The xProof xAI endpoint reconstructs the full picture — WHO (Grok), WHAT (output hash), WHEN (blockchain timestamps), WHY (prompt hash anchored before generation).

In [ ]:
verification = requests.get(
    f"{XPROOF_BASE}/api/xai/grok-cookbook-demo"
).json()

print(json.dumps(verification, indent=2))

## Step 5: Investigate with Incident Report

For any proof, you can pull the full incident report — the complete audit trail reconstruction showing the WHY→WHAT chain, timestamps, and verdict.

This is the same investigation tool that autonomous agents use to audit each other.

In [ ]:
xai_data = requests.get(f"{XPROOF_BASE}/api/xai/grok-cookbook-demo").json()

if xai_data.get("xproof") and xai_data["xproof"].get("wallet"):
    wallet = xai_data["xproof"]["wallet"]
    proof_id = what_proof["id"]
    incident_url = f"{XPROOF_BASE}/incident/{wallet}/{proof_id}"
    print(f"Full incident report: {incident_url}")
    print(f"\nThis page shows:")
    print(f"  - Verdict (clean/anomaly/incomplete)")
    print(f"  - WHO: agent identity + trust score")
    print(f"  - WHY proof: prompt hash anchored BEFORE generation")
    print(f"  - WHAT proof: output hash anchored AFTER generation")
    print(f"  - Timeline with blockchain-verified timestamps")
else:
    print("Run steps 1-3 first to create linked proofs.")

## How It Works Under the Hood

```
┌─────────────┐     ┌──────────┐     ┌───────────────┐
│  Your App    │     │  Grok    │     │   xProof      │
│              │     │  API     │     │   (MultiversX) │
└──────┬───────┘     └────┬─────┘     └───────┬───────┘
       │                  │                   │
       │  1. Hash prompt  │                   │
       │──────────────────────────────────────>│
       │              WHY anchored on-chain   │
       │<─────────────────────────────────────│
       │                  │                   │
       │  2. Send prompt  │                   │
       │─────────────────>│                   │
       │    Grok output   │                   │
       │<─────────────────│                   │
       │                  │                   │
       │  3. Hash output  │                   │
       │──────────────────────────────────────>│
       │             WHAT anchored on-chain   │
       │<─────────────────────────────────────│
       │                  │                   │
       │  4. Anyone can verify:               │
       │     WHY timestamp < WHAT timestamp   │
       │     = intent preceded output          │
```

### Key Properties

- **Tamper-proof**: Hashes are anchored on MultiversX blockchain — can't be altered retroactively
- **Timestamped**: Blockchain provides immutable ordering — WHY always provably before WHAT
- **Public**: Anyone can verify via `GET /api/xai/{agent_id}` — no special access needed
- **Machine-readable**: The API returns structured JSON for automated verification
- **Free to start**: 10 free certifications, then $0.05/proof via USDC on Base (x402 protocol)

## Production Integration Pattern

For production agents, wrap the pattern into a reusable function:

In [ ]:
def verified_grok_call(
    prompt: str,
    agent_id: str = "my-grok-agent",
    model: str = "grok-3",
    system_prompt: str = "You are a helpful assistant.",
) -> dict:
    """Make a Grok API call with full xProof verification.

    Returns the Grok output plus WHY and WHAT proof IDs
    for complete audit trail reconstruction.
    """
    prompt_hash = sha256_hash(prompt)

    why_proof = certify_on_xproof(
        file_hash=prompt_hash,
        filename=f"{agent_id}-intent.txt",
        metadata={
            "xai_agent_id": agent_id,
            "xai_model": model,
            "action_type": "reason",
            "stage": "pre_execution",
            "prompt_hash": prompt_hash,
        },
    )

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )

    output = completion.choices[0].message.content
    output_hash = sha256_hash(output)

    what_proof = certify_on_xproof(
        file_hash=output_hash,
        filename=f"{agent_id}-output.txt",
        metadata={
            "xai_agent_id": agent_id,
            "xai_model": completion.model,
            "action_type": "generate",
            "stage": "post_execution",
            "prompt_hash": prompt_hash,
            "content_hash": output_hash,
        },
    )

    return {
        "output": output,
        "model": completion.model,
        "why_proof_id": why_proof["id"],
        "what_proof_id": what_proof["id"],
        "verify_url": f"{XPROOF_BASE}/api/xai/{agent_id}",
        "incident_report": f"{XPROOF_BASE}/incident/{{wallet}}/{what_proof['id']}",
    }

In [ ]:
result = verified_grok_call(
    prompt="What are the three most important properties a trust score for AI agents should have?",
    agent_id="grok-cookbook-demo",
    model="grok-3",
)

print(f"WHY proof: {result['why_proof_id']}")
print(f"WHAT proof: {result['what_proof_id']}")
print(f"Verify agent: {result['verify_url']}")
print(f"\nGrok output:\n{result['output'][:500]}...")

## Resources

- [xProof API Documentation](https://xproof.app/docs)
- [xProof Machine-Readable Spec](https://xproof.app/.well-known/xproof.md)
- [xAI/Grok Integration Endpoint](https://xproof.app/api/xai/grok-cookbook-demo)
- [Agent Proof Standard](https://xproof.app/api/standard/spec) — open, stack-agnostic proof format
- [xProof on GitHub](https://github.com/jasonxkensei/xProof)

### Pricing

| Method | Cost | Details |
|--------|------|--------|
| Free trial | $0 | 10 certifications, no wallet needed |
| x402 (USDC on Base) | $0.05/proof | Pay per request, no account needed |
| API key | $0.05/proof | Prepaid credits via USDC on Base |

### x402 Payment (No Account Required)

Agents can also pay per-request with USDC on Base — no registration needed:

```bash
# Request without auth returns 402 with payment requirements
curl -X POST https://xproof.app/api/proof \
  -H "Content-Type: application/json" \
  -d '{"fileHash": "abc...", "fileName": "test.txt"}'

# Response: {"x402Version": 1, "accepts": [{"price": "$0.05", "network": "eip155:8453", ...}]}
```